# Model 6.3 no-time inference template

This notebook reproduces the public preprocessing, five-fold ensemble inference, canonical Accumulative Inference Curve (AIC) calculation, and paper-style result plot. The default record is synthetic and demonstrates only the input format. Replace the manifest path with your own data for research use.

The first run loads about 475 MiB of model artifacts sequentially and evaluates 18 azimuthal variants with each fold. Progress is printed by record, fold, variation, and batch.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src" / "liquefaction_timing").is_dir():
    raise FileNotFoundError(
        "Run this notebook from the repository root or notebooks folder."
    )
sys.path.insert(0, str(REPO_ROOT / "src"))

from liquefaction_timing import (
    build_feature_variations,
    load_manifest,
    public_configuration,
    run_manifest,
)

print(f"Repository root: {REPO_ROOT}")

## Configure paths

Edit `MANIFEST_PATH` to point to your manifest. Relative motion paths inside the manifest are resolved from the manifest's folder. `OUTPUT_ROOT` can be changed freely. Do not change `MODEL_ROOT` unless you deliberately relocate the supplied model artifacts.

In [ ]:
MANIFEST_PATH = REPO_ROOT / "data" / "example" / "input_manifest.csv"
MODEL_ROOT = REPO_ROOT / "models" / "model6_3_no_time"
OUTPUT_ROOT = REPO_ROOT / "outputs"

PREDICTION_BATCH_SIZE = 256  # Reduce to 128 or 64 if GPU memory is limited.
SKIP_EXISTING = True

display(pd.read_csv(MANIFEST_PATH))
public_configuration()

## Validate preprocessing before inference

This cell loads the records, verifies their units-independent structural requirements, resamples when needed, creates all 18 rotations, and confirms the final six-channel dimensions. It does not load TensorFlow or run the model.

In [ ]:
records = load_manifest(MANIFEST_PATH)
for record in records:
    variations = build_feature_variations(record)
    print(
        record.recording_id,
        f"samples={len(record.time_s):,}",
        f"duration={record.time_s[-1]:.3f} s",
        f"variations={len(variations)}",
        f"feature_shape={variations[0].shape}",
        f"Mw={record.mw}",
    )
del variations

## Run the five-fold ensemble and AIC post-processing

Outputs are saved immediately after each parent record. The median hard-count Class-1 AIC and threshold of 100 determine whether triggering is detected. For a detected record, the median accumulated Class-1 softmax probability curve determines `tL`. A record below the hard-count threshold receives no `tL`.

In [ ]:
summary = run_manifest(
    manifest_path=MANIFEST_PATH,
    model_root=MODEL_ROOT,
    output_root=OUTPUT_ROOT,
    batch_size=PREDICTION_BATCH_SIZE,
    display_plots=True,
    skip_existing=SKIP_EXISTING,
)
display(summary)

## Output locations

Each record receives `summary.json`, `ensemble_predictions.npz`, `aic_curves.csv`, and `inference_plot.png` under `OUTPUT_ROOT/<recording_id>/`. The output root also contains `inference_summary.csv`. The `.npz` file stores five-fold-averaged softmax probabilities with shape `[18, number_of_windows, 3]`.